# 08 — RAG on Your Own Data

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Drop your own PDF / Excel / CSV / text files into `data/user_data/`.
2. Auto-detect, chunk, embed, and build a private vector store.
3. Ask questions and inspect citations against *your* documents.
4. Apply confidentiality safeguards.


## ⚠️ Before you add files — read this

1. **Anonymise.** Remove or mask client names, PANs, employee IDs.
2. **Engagement-letter authority.** Confirm AI use is permitted.
3. **Set `EMBEDDING_PROVIDER=local` in `.env`** so embeddings stay on your laptop.
4. **Switch `LLM_PROVIDER=mock`** if you want to test retrieval *without* sending chunks to a public LLM.
5. Files in `data/user_data/` are ignored by git by default. Keep it that way.


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


## 8.1 — Detect what's available

In [ ]:
from pathlib import Path
user_dir = Path('data/user_data')
files = [p for p in user_dir.rglob('*') if p.is_file() and p.suffix.lower() in ('.pdf','.xlsx','.xls','.csv','.txt','.md')]
print(f'Found {len(files)} supported file(s):')
for f in files:
    print('  -', f.relative_to(user_dir))
if not files:
    print('\nNo files yet. Falling back to the synthetic dataset so the lesson still runs.')

## 8.2 — Build the store (auto-fallback to synthetic data)

In [ ]:
from src.rag_utils import build_store_from_folder
from pathlib import Path

source_folder = 'data/user_data' if any(Path('data/user_data').rglob('*.*')) else 'data/generated/pdf'
print('Using folder:', source_folder)
store = build_store_from_folder(source_folder, chunk_size=800, overlap=120)
print(f'Store size: {len(store)} chunks')

## 8.3 — Configure your questions

In [ ]:
# Edit this list with questions about your own data
your_questions = [
    'Summarise the most important findings.',
    'What are the key risks?',
    'Are there any policy violations?',
    'Are PAN/VAT numbers consistently captured?',
]
from src.rag_utils import rag_answer
for q in your_questions:
    answer, hits = rag_answer(q, store, k=4, return_hits=True)
    print('Q:', q)
    print('A:', answer)
    print('Sources:', [(h['metadata'].get('source'), h['metadata'].get('page')) for h in hits])
    print('-' * 80)

## 8.4 — Save the store to disk (optional, persistent)

In [ ]:
# Persist a Chroma-backed store so we don't re-embed every notebook reload.
from src.rag_utils import build_store_from_folder
try:
    persistent = build_store_from_folder(source_folder, persist_dir='outputs/vector_store/user_data')
    print('Persisted store size:', len(persistent))
except Exception as e:
    print('Skipping Chroma persistence:', e)

## Expected output

* The detect cell lists every supported file in `data/user_data/`.
* If you added nothing, the notebook silently falls back to the synthetic PDFs.


## Exercise

1. Drop one *anonymised* policy document and ask 3 policy-compliance questions.
2. Drop a trial-balance Excel and ask: *"Which expense lines show unusual growth versus last year?"*
3. With a `doc_type` filter, restrict retrieval to only the policy file.


## Common errors

| Symptom | Fix |
|---|---|
| `[warn] Skipping ...: ParserError` | The file is corrupt or password-protected. |
| Slow embedding | Large files — increase `chunk_size`, or limit files in `data/user_data/`. |
| OCR-needed PDF returns nothing | The PDF is a scan; OCR is required (covered in Notebook 09). |


## ⚠️ Professional caution (repeat)

*Anything you put in `data/user_data/` and ask a public LLM about will be sent to that LLM.* Use the **mock** or **local** providers if confidentiality matters and you have no engagement authority. Delete the files after the session if they were a one-off.